# Test RAGAS

In [1]:
import sys
sys.path.insert(0, '/home/local/QCRI/fdeniz/projects/sspbench')

import importlib
import sspbench.novelty.ragas_utils
importlib.reload(sspbench.novelty.ragas_utils)

from sspbench.novelty.llm_utils import create_model_from_config
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness,
    SentenceTransformerEmbeddings
)

print("✓ Imports successful")
print(f"RAGAS available: {is_ragas_available()}")

INFO 02-05 10:05:02 [__init__.py:216] Automatically detected platform cuda.


/home/local/QCRI/fdeniz/anaconda3/envs/autobencher/lib/python3.10/site-packages/flaml/__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


✓ Imports successful
RAGAS available: True


## Setup Eval Model

In [2]:
import os
# Set dummy OpenAI API key to prevent RAGAS from requiring real OpenAI credentials
os.environ["OPENAI_API_KEY"] = "dummy-key-for-ragas"

eval_config = {
    "type": "openai",
    "model": "gpt-oss",
    "api_url": "http://10.4.8.217:8000/v1",
    "api_token": "abc123",
    "api_version": "2024-12-01-preview"
}

try:
    eval_model = create_model_from_config(eval_config)
    print(f"✓ eval_model created successfully: {type(eval_model)}. Sample response: {eval_model.generate('Hello')}")
except Exception as e:
    print(f"✗ Failed to create eval_model: {e}")
    eval_model = None

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

Loading new model: gpt-oss
[Warning] Generation config not defined: {}, using defaults.
Discovered supported parameters: ['max_tokens', 'temperature', 'top_p', 'presence_penalty']
Generated Model: OpenaiLLM Fail on empty response: False
Content in batch was blocked by API model, trying chat inference...
✓ eval_model created successfully: <class 'models.openai_model.OpenaiLLM'>. Sample response: ['Hello! How can I assist you today?']


In [3]:
# Sample paragraph for testing
test_paragraph = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.
""".strip()

print("Test paragraph:")
print(test_paragraph)
print("\n" + "="*80 + "\n")

Test paragraph:
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.




In [4]:
# Configure RAGAS synthesizers for shorter answers
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
from ragas.testset import TestsetGenerator
from sspbench.novelty.ragas_utils import CustomRagasLLM, CustomRagasEmbeddings

# Create RAGAS adapters
ragas_llm = CustomRagasLLM(eval_model, temperature=0.0)
ragas_embeddings = CustomRagasEmbeddings(embedding_model)

# Configure query distribution for ONLY short, specific questions
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),  # 100% single-hop specific for shortest answers
]

print("✓ Configured query distribution for short answers:")
for synthesizer, weight in query_distribution:
    print(f"  {synthesizer.__class__.__name__}: {weight}")
print()

✓ Configured query distribution for short answers:
  SingleHopSpecificQuerySynthesizer: 1.0



In [5]:
if eval_model and is_ragas_available():
    print("Generating Q&A pairs with RAGAS using synthesizers...\n")
    try:
        # Create generator with synthesizer configuration
        generator = TestsetGenerator(
            llm=ragas_llm,
            embedding_model=ragas_embeddings,
        )

        from langchain_core.documents import Document
        doc = Document(page_content=test_paragraph)

        query_distribution = [
            (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),
        ]
        testset = generator.generate_with_langchain_docs(
            documents=[doc],
            testset_size=3,
            query_distribution=query_distribution
        )

        qa_pairs = []
        for idx, sample in enumerate(testset.samples, 1):
            qa_pairs.append(
                {
                    "id": str(idx),
                    "question": sample.eval_sample.user_input,
                    "answer": sample.eval_sample.reference,
                }
            )

        print(f"✓ Generated {len(qa_pairs)} Q&A pairs:\n")
        for i, qa in enumerate(qa_pairs, 1):
            print(f"Q{i}: {qa['question']}")
            print(f"A{i}: {qa['answer']}")
            print("-" * 80)
    except Exception as e:
        print(f"✗ Error generating Q&A: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

Generating Q&A pairs with RAGAS using synthesizers...



Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Generated 3 Q&A pairs:

Q1: What is the significance of the Eiffel Tower to France?
A1: The Eiffel Tower, a wrought‑iron lattice tower on the Champ de Mars in Paris, France, was constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair. Although it was initially criticized by some of France's leading artists and intellectuals for its design, it has become a global cultural icon of France and one of the most recognizable structures in the world.
--------------------------------------------------------------------------------
Q2: I wanna know why the Eiffel Tower in Paris was put there, who built it, when it was built, what was it for, and why people at first didn't like it but now everybody knows it all over the world?
A2: The Eiffel Tower is a wrought‑iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. It was constructed from 1887 to 1889 as the entrance arch to the 1889

## Evaluate Q&A Pairs with DirectFaithfulnessEvaluator

In [6]:
# Evaluate each Q&A pair using DirectFaithfulnessEvaluator
if 'qa_pairs' in locals() and qa_pairs:
    print("Evaluating Q&A pairs using DirectFaithfulnessEvaluator...\n")

    from sspbench.evaluators import DirectFaithfulnessEvaluator

    # Create evaluator instance
    direct_evaluator = DirectFaithfulnessEvaluator(eval_model=eval_model)

    # Prepare samples with context
    samples = []
    for qa in qa_pairs:
        samples.append({
            "id": qa["id"],
            "question": qa["question"],
            "answer": qa["answer"],
            "context": test_paragraph
        })

    # Evaluate using the evaluator
    results = direct_evaluator.evaluate(samples)

    # Display results
    for result in results["results"]:
        print(f"Evaluating Q&A pair {result['sample_id']}:")
        print(f"Question: {result['question']}")
        print(f"Answer: {result['answer']}")
        print(f"Faithfulness: {result['faithfulness']:.4f}")
        print(f"Answer Relevancy: {result['answer_relevancy']:.4f}")
        print("-" * 80)

    # Display summary
    summary = results["summary"]
    print("Summary:")
    print(f"Average Faithfulness: {summary['average_faithfulness']:.4f}")
    print(f"Average Answer Relevancy: {summary['average_relevancy']:.4f}")
    print(f"Total Samples: {summary['total_samples']}")

else:
    print("⚠️ No qa_pairs found. Please run the Q&A generation cell first.")

Evaluating Q&A pairs using DirectFaithfulnessEvaluator...

Evaluating Q&A pair 1:
Question: What is the significance of the Eiffel Tower to France?
Answer: The Eiffel Tower, a wrought‑iron lattice tower on the Champ de Mars in Paris, France, was constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair. Although it was initially criticized by some of France's leading artists and intellectuals for its design, it has become a global cultural icon of France and one of the most recognizable structures in the world.
Faithfulness: 1.0000
Answer Relevancy: 1.0000
--------------------------------------------------------------------------------
Evaluating Q&A pair 2:
Question: I wanna know why the Eiffel Tower in Paris was put there, who built it, when it was built, what was it for, and why people at first didn't like it but now everybody knows it all over the world?
Answer: The Eiffel Tower is a wrought‑iron lattice tower on the Champ de Mars in Paris, France. It is named aft

In [7]:
# Evaluate Q&A pairs using RagasFaithfulnessEvaluator
if 'qa_pairs' in locals() and qa_pairs and eval_model is not None:
    print("Evaluating Q&A pairs using RagasFaithfulnessEvaluator...\n")

    from sspbench.evaluators import RagasFaithfulnessEvaluator

    # Create evaluator instance
    ragas_evaluator = RagasFaithfulnessEvaluator(
        eval_model=eval_model,
        embedding_model=embedding_model
    )

    # Prepare samples with context
    samples = []
    for qa in qa_pairs:
        samples.append({
            "id": qa["id"],
            "question": qa["question"],
            "answer": qa["answer"],
            "context": test_paragraph
        })

    # Evaluate using the evaluator
    results = ragas_evaluator.evaluate(samples)

    # Display results
    for result in results["results"]:
        print(f"Evaluating Q&A pair {result['sample_id']} (RAGAS):")
        print(f"Question: {result['question']}")
        print(f"Answer: {result['answer']}")
        print(f"Faithfulness: {result['faithfulness']:.4f}")
        print(f"Answer Relevancy: {result['answer_relevancy']:.4f}")
        print("-" * 80)

    # Display summary
    summary = results["summary"]
    print("Summary:")
    print(f"Average Faithfulness: {summary['average_faithfulness']:.4f}")
    print(f"Average Answer Relevancy: {summary['average_relevancy']:.4f}")
    print(f"Total Samples: {summary['total_samples']}")

else:
    print("⚠️ No qa_pairs found or eval_model not available. Please run the Q&A generation cell first.")

Evaluating Q&A pairs using RagasFaithfulnessEvaluator...



LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating Q&A pair 1 (RAGAS):
Question: What is the significance of the Eiffel Tower to France?
Answer: The Eiffel Tower, a wrought‑iron lattice tower on the Champ de Mars in Paris, France, was constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair. Although it was initially criticized by some of France's leading artists and intellectuals for its design, it has become a global cultural icon of France and one of the most recognizable structures in the world.
Faithfulness: 1.0000
Answer Relevancy: 0.7987
--------------------------------------------------------------------------------
Evaluating Q&A pair 2 (RAGAS):
Question: I wanna know why the Eiffel Tower in Paris was put there, who built it, when it was built, what was it for, and why people at first didn't like it but now everybody knows it all over the world?
Answer: The Eiffel Tower is a wrought‑iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose compa

## Comparison: DirectFaithfulnessEvaluator vs RagasFaithfulnessEvaluator

This notebook compares the results from:
- **Cell 10**: DirectFaithfulnessEvaluator using custom prompts with your model
- **Cell 12**: RagasFaithfulnessEvaluator using RAGAS library metrics

Both evaluators use your custom model but different evaluation approaches.

## Summary

This notebook demonstrates:
1. ✓ RAGAS setup with SingleHopSpecificQuerySynthesizer
2. ✓ Q&A pair generation with faithfulness evaluation for each pair
3. ✓ Comparison between DirectFaithfulnessEvaluator and RagasFaithfulnessEvaluator
4. ✓ Clean, focused testing of the evaluator classes

Each generated Q&A pair includes faithfulness and answer relevancy scores from both evaluation methods.